In [112]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
)

from xgboost import XGBClassifier

In [113]:
DATA_PATH = "transactions_dataset.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

payments = raw["Response"]

rows = []

for item in payments:
    p = item["Payment"]

    counterparty = p.get("counterparty_alias", {}) or {}
    counterparty_label = counterparty.get("label_user", {}) or {}

    alias = p.get("alias", {}) or {}
    alias_label = alias.get("label_user", {}) or {}

    payment_arrival = p.get("payment_arrival_expected", {}) or {}

    rows.append({
        "payment_id": p.get("id"),
        "created": p.get("created"),
        "updated": p.get("updated"),
        "monetary_account_id": p.get("monetary_account_id"),

        "amount_value": float(p["amount"]["value"]),
        "amount_currency": p["amount"]["currency"],

        # Keep text fields in df for inspection only.
        # Do NOT use them as features.
        "description": p.get("description", ""),
        "counterparty_display_name": counterparty.get("display_name"),

        "payment_type": p.get("type"),
        "sub_type": p.get("sub_type"),
        "merchant_reference": p.get("merchant_reference"),

        "own_display_name": alias.get("display_name"),
        "own_iban": alias.get("iban"),
        "own_user_type": alias_label.get("type"),

        "counterparty_iban": counterparty.get("iban"),
        "counterparty_country": counterparty.get("country"),
        "counterparty_user_type": counterparty_label.get("type"),

        "attachment_count": len(p.get("attachment", []) or []),
        "payment_arrival_status": payment_arrival.get("status"),

        "balance_after": float(p["balance_after_mutation"]["value"]),

        # Labels
        "is_scam": int(p.get("_is_synthetic_fraud", False)),
        "scam_type": p.get("_synthetic_scam_type", "none"),
    })

df = pd.DataFrame(rows)

df.head()

,payment_id,created,updated,monetary_account_id,amount_value,amount_currency,description,counterparty_display_name,payment_type,sub_type,...,own_iban,own_user_type,counterparty_iban,counterparty_country,counterparty_user_type,attachment_count,payment_arrival_status,balance_after,is_scam,scam_type
0,13356886,2026-01-01 19:08:47.107473,2026-01-01 19:08:47.107473,1728977,-473.99,EUR,Administratiekosten 108246,Jumbo,MASTERCARD,PAYMENT,...,NL39KNAB5935868570,PERSON,NL41ASNB2633229661,NL,COMPANY,0,ARRIVED,1932.04,0,none
1,53524491,2025-12-29 18:52:02.844151,2025-12-29 18:52:02.844151,7073292,-44.47,EUR,Achterstand zorgverzekering,PayPal Europe,IDEAL,PAYMENT,...,NL86INGB9147811653,PERSON,NL23SNSB7321077975,NL,COMPANY,0,ARRIVED,3285.79,0,none
2,81182864,2025-11-11 22:31:01.117301,2025-11-11 22:31:01.117301,8099076,4014.89,EUR,Huur appartement,Coolblue,BUNQ,PAYMENT,...,NL18INGB9767363842,PERSON,NL83BUNQ0629802621,NL,COMPANY,0,ARRIVED,2880.67,0,none
3,66851760,2026-03-29 17:43:41.677568,2026-03-29 17:43:41.677568,1247595,-36.65,EUR,Tikkie drankjes,Airbnb Payments,BUNQ,PAYMENT,...,NL68ABNA9814240392,PERSON,NL75INGB2955705581,NL,COMPANY,0,ARRIVED,7309.67,0,none
4,86460539,2026-04-05 10:39:05.439589,2026-04-05 10:39:05.439589,9487336,-849.99,EUR,Zorgverzekering,F. Jacobs,BUNQ,PAYMENT,...,NL77SNSB1302245398,PERSON,NL16SNSB1152672624,NL,PERSON,1,ARRIVED,-96.25,0,none


In [114]:
print(df.shape)

print("\nBinary label distribution:")
print(df["is_scam"].value_counts())
print(df["is_scam"].value_counts(normalize=True))

print("\nScam type distribution:")
print(df["scam_type"].value_counts())

(5000, 22)

Binary label distribution:
is_scam
0    4730
1     270
Name: count, dtype: int64
is_scam
0    0.946
1    0.054
Name: proportion, dtype: float64

Scam type distribution:
scam_type
none                      4730
fake_invoice                57
marketplace_scam            54
phishing                    53
whatsapp_impersonation      53
gift_card                   35
investment_scam             18
Name: count, dtype: int64


In [115]:
def extract_bank_code(iban):
    if not isinstance(iban, str):
        return None

    if len(iban) >= 8:
        return iban[4:8]

    return None


df_fe = df.copy()

df_fe["created_dt"] = pd.to_datetime(df_fe["created"], errors="coerce")

df_fe["created_hour"] = df_fe["created_dt"].dt.hour
df_fe["created_dayofweek"] = df_fe["created_dt"].dt.dayofweek
df_fe["created_month"] = df_fe["created_dt"].dt.month
df_fe["is_weekend"] = df_fe["created_dayofweek"].isin([5, 6]).astype(int)
df_fe["is_night"] = df_fe["created_hour"].between(0, 5).astype(int)

df_fe["amount_abs"] = df_fe["amount_value"].abs()
df_fe["is_outgoing"] = (df_fe["amount_value"] < 0).astype(int)
df_fe["is_incoming"] = (df_fe["amount_value"] > 0).astype(int)

df_fe["balance_after_abs"] = df_fe["balance_after"].abs()
df_fe["amount_balance_ratio"] = df_fe["amount_abs"] / (df_fe["balance_after_abs"] + 1)

df_fe["has_merchant_reference"] = df_fe["merchant_reference"].notna().astype(int)
df_fe["has_attachment"] = (df_fe["attachment_count"] > 0).astype(int)

df_fe["counterparty_bank_code"] = df_fe["counterparty_iban"].apply(extract_bank_code)
df_fe["own_bank_code"] = df_fe["own_iban"].apply(extract_bank_code)

df_fe.head()

,payment_id,created,updated,monetary_account_id,amount_value,amount_currency,description,counterparty_display_name,payment_type,sub_type,...,is_night,amount_abs,is_outgoing,is_incoming,balance_after_abs,amount_balance_ratio,has_merchant_reference,has_attachment,counterparty_bank_code,own_bank_code
0,13356886,2026-01-01 19:08:47.107473,2026-01-01 19:08:47.107473,1728977,-473.99,EUR,Administratiekosten 108246,Jumbo,MASTERCARD,PAYMENT,...,0,473.99,1,0,1932.04,0.245204,1,0,ASNB,KNAB
1,53524491,2025-12-29 18:52:02.844151,2025-12-29 18:52:02.844151,7073292,-44.47,EUR,Achterstand zorgverzekering,PayPal Europe,IDEAL,PAYMENT,...,0,44.47,1,0,3285.79,0.013530,0,0,SNSB,INGB
2,81182864,2025-11-11 22:31:01.117301,2025-11-11 22:31:01.117301,8099076,4014.89,EUR,Huur appartement,Coolblue,BUNQ,PAYMENT,...,0,4014.89,0,1,2880.67,1.393251,0,0,BUNQ,INGB
3,66851760,2026-03-29 17:43:41.677568,2026-03-29 17:43:41.677568,1247595,-36.65,EUR,Tikkie drankjes,Airbnb Payments,BUNQ,PAYMENT,...,0,36.65,1,0,7309.67,0.005013,0,0,INGB,ABNA
4,86460539,2026-04-05 10:39:05.439589,2026-04-05 10:39:05.439589,9487336,-849.99,EUR,Zorgverzekering,F. Jacobs,BUNQ,PAYMENT,...,0,849.99,1,0,96.25,8.740257,0,1,SNSB,SNSB


In [116]:
numeric_features = [
    "amount_value",
    "amount_abs",
    "balance_after",
    "balance_after_abs",
    "amount_balance_ratio",

    "created_hour",
    "created_dayofweek",
    "created_month",
    "is_weekend",
    "is_night",

    "is_outgoing",
    "is_incoming",

    "attachment_count",
    "has_attachment",
    "has_merchant_reference",
]

categorical_features = [
    "payment_type",
    "sub_type",
    "amount_currency",
    "counterparty_country",
    "counterparty_user_type",
    "counterparty_bank_code",
    "own_bank_code",
    "payment_arrival_status",
]

feature_cols = numeric_features + categorical_features

X = df_fe[feature_cols].copy()
y_binary = df_fe["is_scam"].copy()
y_type = df_fe["scam_type"].copy()

X.head()

,amount_value,amount_abs,balance_after,balance_after_abs,amount_balance_ratio,created_hour,created_dayofweek,created_month,is_weekend,is_night,...,has_attachment,has_merchant_reference,payment_type,sub_type,amount_currency,counterparty_country,counterparty_user_type,counterparty_bank_code,own_bank_code,payment_arrival_status
0,-473.99,473.99,1932.04,1932.04,0.245204,19,3,1,0,0,...,0,1,MASTERCARD,PAYMENT,EUR,NL,COMPANY,ASNB,KNAB,ARRIVED
1,-44.47,44.47,3285.79,3285.79,0.013530,18,0,12,0,0,...,0,0,IDEAL,PAYMENT,EUR,NL,COMPANY,SNSB,INGB,ARRIVED
2,4014.89,4014.89,2880.67,2880.67,1.393251,22,1,11,0,0,...,0,0,BUNQ,PAYMENT,EUR,NL,COMPANY,BUNQ,INGB,ARRIVED
3,-36.65,36.65,7309.67,7309.67,0.005013,17,6,3,1,0,...,0,0,BUNQ,PAYMENT,EUR,NL,COMPANY,INGB,ABNA,ARRIVED
4,-849.99,849.99,-96.25,96.25,8.740257,10,6,4,1,0,...,1,0,BUNQ,PAYMENT,EUR,NL,PERSON,SNSB,SNSB,ARRIVED


In [117]:
forbidden_patterns = [
    "fraud",
    "scam",
    "description",
    "display_name",
    "name",
    "label",
]

bad_cols = []

for col in X.columns:
    lowered = col.lower()
    if any(pattern in lowered for pattern in forbidden_patterns):
        bad_cols.append(col)

bad_cols

[]

In [118]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_binary,
    test_size=0.25,
    random_state=42,
    stratify=y_binary,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train scam rate:", y_train.mean())
print("Test scam rate:", y_test.mean())

Train shape: (3750, 23)
Test shape: (1250, 23)
Train scam rate: 0.05413333333333333
Test scam rate: 0.0536


In [119]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

In [120]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

binary_xgb_model = XGBClassifier(
    n_estimators=350,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
)

binary_model_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", binary_xgb_model),
    ]
)

binary_model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['amount_value', 'amount_abs',
                                                   'balance_after',
                                                   'balance_after_abs',
                                                   'amount_balance_ratio',
                                                   'created_hour',
                                                   'created_dayofweek',
                                                   'created_month',
                                                   'is_weekend', 'is_night',
                                                   'is_outgoing', 'is_incoming',
                                                   'attachment_count',
                                                   'has_attachment',
                                                   'has_m...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=350, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [121]:
y_pred = binary_model_pipeline.predict(X_test)
y_proba = binary_model_pipeline.predict_proba(X_test)[:, 1]

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

Confusion matrix:
[[963 220]
 [ 52  15]]

Classification report:
              precision    recall  f1-score   support

           0       0.95      0.81      0.88      1183
           1       0.06      0.22      0.10        67

    accuracy                           0.78      1250
   macro avg       0.51      0.52      0.49      1250
weighted avg       0.90      0.78      0.83      1250

ROC-AUC: 0.6432924136712885
PR-AUC: 0.08509620437228545


In [122]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]

rows = []

for threshold in thresholds:
    y_pred_t = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    rows.append({
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "precision": precision_score(y_test, y_pred_t, zero_division=0),
        "recall": recall_score(y_test, y_pred_t, zero_division=0),
        "f1": f1_score(y_test, y_pred_t, zero_division=0),
        "false_positive_rate": fp / (fp + tn),
    })

threshold_df = pd.DataFrame(rows)
threshold_df

,threshold,TP,FP,FN,TN,precision,recall,f1,false_positive_rate
0,0.10,60,825,7,358,0.067797,0.895522,0.126050,0.697380
1,0.15,55,714,12,469,0.071521,0.820896,0.131579,0.603550
2,0.20,50,618,17,565,0.074850,0.746269,0.136054,0.522401
3,0.25,47,533,20,650,0.081034,0.701493,0.145286,0.450549
4,0.30,40,441,27,742,0.083160,0.597015,0.145985,0.372781
5,0.40,28,335,39,848,0.077135,0.417910,0.130233,0.283178
6,0.50,15,220,52,963,0.063830,0.223881,0.099338,0.185968
7,0.60,13,124,54,1059,0.094891,0.194030,0.127451,0.104818
8,0.70,4,49,63,1134,0.075472,0.059701,0.066667,0.041420


In [123]:
df_scam = df_fe[df_fe["is_scam"] == 1].copy()

X_type = df_scam[feature_cols].copy()
y_type_raw = df_scam["scam_type"].copy()

print("Scam-only shape:", X_type.shape)
print(y_type_raw.value_counts())

Scam-only shape: (270, 23)
scam_type
fake_invoice              57
marketplace_scam          54
phishing                  53
whatsapp_impersonation    53
gift_card                 35
investment_scam           18
Name: count, dtype: int64


In [124]:
type_label_encoder = LabelEncoder()

y_type_encoded = type_label_encoder.fit_transform(y_type_raw)

print("Classes:")
for index, class_name in enumerate(type_label_encoder.classes_):
    print(index, "->", class_name)

Classes:
0 -> fake_invoice
1 -> gift_card
2 -> investment_scam
3 -> marketplace_scam
4 -> phishing
5 -> whatsapp_impersonation


In [125]:
X_type_train, X_type_test, y_type_train, y_type_test = train_test_split(
    X_type,
    y_type_encoded,
    test_size=0.25,
    random_state=42,
    stratify=y_type_encoded,
)

print("Train shape:", X_type_train.shape)
print("Test shape:", X_type_test.shape)

Train shape: (202, 23)
Test shape: (68, 23)


In [126]:
type_preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

num_classes = len(type_label_encoder.classes_)

type_xgb_model = XGBClassifier(
    n_estimators=350,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=num_classes,
    random_state=42,
    n_jobs=-1,
)

type_model_pipeline = Pipeline(
    steps=[
        ("preprocessor", type_preprocessor),
        ("model", type_xgb_model),
    ]
)

type_model_pipeline.fit(X_type_train, y_type_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['amount_value', 'amount_abs',
                                                   'balance_after',
                                                   'balance_after_abs',
                                                   'amount_balance_ratio',
                                                   'created_hour',
                                                   'created_dayofweek',
                                                   'created_month',
                                                   'is_weekend', 'is_night',
                                                   'is_outgoing', 'is_incoming',
                                                   'attachment_count',
                                                   'has_attachment',
                                                   'has_m...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.05,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=350, n_jobs=-1, num_class=6, ...))])

In [ ]:
y_type_pred = type_model_pipeline.predict(f)
y_type_proba = type_model_pipeline.predict_proba(X_type_test)

print("Confusion matrix:")
print(confusion_matrix(y_type_test, y_type_pred))

print("\nClassification report:")
print(
    classification_report(
        y_type_test,
        y_type_pred,
        target_names=type_label_encoder.classes_,
        zero_division=0,
    )
)

Confusion matrix:
[[5 2 2 3 2 0]
 [2 5 0 0 2 0]
 [2 0 3 0 0 0]
 [0 2 0 6 0 6]
 [1 1 0 4 7 0]
 [2 1 1 4 2 3]]

Classification report:
                        precision    recall  f1-score   support

          fake_invoice       0.42      0.36      0.38        14
             gift_card       0.45      0.56      0.50         9
       investment_scam       0.50      0.60      0.55         5
      marketplace_scam       0.35      0.43      0.39        14
              phishing       0.54      0.54      0.54        13
whatsapp_impersonation       0.33      0.23      0.27        13

              accuracy                           0.43        68
             macro avg       0.43      0.45      0.44        68
          weighted avg       0.42      0.43      0.42        68



In [128]:
SCAM_THRESHOLD = 0.40


def predict_payment_risk(payment_features_df):
    """
    payment_features_df must already contain the same feature columns as X:
    numeric_features + categorical_features.

    Returns:
    - boolean scam decision
    - scam probability
    - optional scam type
    - optional scam type probability
    """

    scam_probability = binary_model_pipeline.predict_proba(payment_features_df)[:, 1]

    results = []

    for i, prob in enumerate(scam_probability):
        is_scam = bool(prob >= SCAM_THRESHOLD)

        if is_scam:
            row = payment_features_df.iloc[[i]]

            type_probas = type_model_pipeline.predict_proba(row)[0]
            type_index = int(np.argmax(type_probas))

            scam_type = type_label_encoder.inverse_transform([type_index])[0]
            scam_type_probability = float(type_probas[type_index])
        else:
            scam_type = None
            scam_type_probability = None

        results.append({
            "is_scam": is_scam,
            "scam_probability": float(prob),
            "scam_type": scam_type,
            "scam_type_probability": scam_type_probability,
        })

    return results

In [129]:
sample_payments = X_test.head(10).copy()

predictions = predict_payment_risk(sample_payments)

pd.DataFrame(predictions)

,is_scam,scam_probability,scam_type,scam_type_probability
0,True,0.747650,phishing,0.543367
1,False,0.006082,None,NaN
2,False,0.387882,None,NaN
3,False,0.121782,None,NaN
4,False,0.213148,None,NaN
5,False,0.112578,None,NaN
6,True,0.568155,phishing,0.622621
7,False,0.209579,None,NaN
8,False,0.266607,None,NaN
9,True,0.448582,whatsapp_impersonation,0.909554


In [130]:
comparison = X_test.head(10).copy()
comparison["true_is_scam"] = y_test.head(10).values

pred_df = pd.DataFrame(predictions)

pd.concat(
    [
        comparison.reset_index(drop=True),
        pred_df.reset_index(drop=True),
    ],
    axis=1
)

,amount_value,amount_abs,balance_after,balance_after_abs,amount_balance_ratio,created_hour,created_dayofweek,created_month,is_weekend,is_night,...,counterparty_country,counterparty_user_type,counterparty_bank_code,own_bank_code,payment_arrival_status,true_is_scam,is_scam,scam_probability,scam_type,scam_type_probability
0,-127.24,127.24,3682.69,3682.69,0.034541,19,6,6,1,0,...,NL,COMPANY,BUNQ,TRIO,ARRIVED,0,True,0.747650,phishing,0.543367
1,3744.37,3744.37,1459.31,1459.31,2.564093,1,3,2,0,1,...,NL,COMPANY,TRIO,KNAB,ARRIVED,0,False,0.006082,None,NaN
2,-65.95,65.95,1845.21,1845.21,0.035722,5,2,2,0,1,...,NL,COMPANY,RABO,KNAB,ARRIVED,0,False,0.387882,None,NaN
3,-24.83,24.83,695.16,695.16,0.035667,11,6,3,1,0,...,NL,PERSON,TRIO,ABNA,ARRIVED,0,False,0.121782,None,NaN
4,-16.24,16.24,1386.98,1386.98,0.011700,18,5,8,1,0,...,NL,COMPANY,RABO,TRIO,ARRIVED,0,False,0.213148,None,NaN
5,-18.63,18.63,2086.65,2086.65,0.008924,4,6,5,1,1,...,NL,COMPANY,RABO,RABO,ARRIVED,0,False,0.112578,None,NaN
6,-886.97,886.97,3168.89,3168.89,0.279811,20,3,1,0,0,...,NL,COMPANY,RABO,BUNQ,ARRIVED,0,True,0.568155,phishing,0.622621
7,-41.16,41.16,18947.21,18947.21,0.002172,10,4,1,0,0,...,NL,COMPANY,RABO,SNSB,ARRIVED,0,False,0.209579,None,NaN
8,-221.43,221.43,69.33,69.33,3.148443,12,6,7,1,0,...,NL,COMPANY,SNSB,TRIO,ARRIVED,0,False,0.266607,None,NaN
9,-596.38,596.38,7214.57,7214.57,0.082652,21,4,10,0,0,...,NL,PERSON,RABO,TRIO,ARRIVED,0,True,0.448582,whatsapp_impersonation,0.909554


In [131]:
joblib.dump(binary_model_pipeline, "xgboost_binary_scam_model.joblib")
joblib.dump(type_model_pipeline, "xgboost_scam_type_model.joblib")
joblib.dump(type_label_encoder, "scam_type_label_encoder.joblib")

print("Saved:")
print("- xgboost_binary_scam_model.joblib")
print("- xgboost_scam_type_model.joblib")
print("- scam_type_label_encoder.joblib")

Saved:
- xgboost_binary_scam_model.joblib
- xgboost_scam_type_model.joblib
- scam_type_label_encoder.joblib
